# CANVASAI : AI Image Enhancer and Outpainter
An AI-powered image enhancement and outpainting tool. Upload any photo — CanvasAI either sharpens and upscales it using Real-ESRGAN, or extends its borders with contextually generated content using Stable Diffusion inpainting.

# Install all Libraries

* gradio       → the UI framework + built-in public URL 
* diffusers    → runs Stable Diffusion inpainting model
* transformers → core Hugging Face library, used by BLIP caption model
* accelerate   → helps models load faster and use GPU memory efficiently
* pillow       → PIL: open, resize, crop, paste, save images
* numpy        → numerical operations — we use it to create the mask (black/white image)             a mask is just a 2D array of 0s and 255s — numpy is perfect for this
* basicsr      → base library that Real-ESRGAN is built on top of
* facexlib     → face detection library, required by Real-ESRGAN internally
* gfpgan       → face enhancement library, also required by Real-ESRGAN internally

In [1]:
!pip install -q basicsr facexlib gfpgan
!pip install -q git+https://github.com/xinntao/Real-ESRGAN.git
!pip install -q gradio diffusers transformers accelerate pillow numpy
print("all libraries installed")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.5/172.5 kB 4.3 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 11.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.2/52.2 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.3/338.3 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 77.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.2/256.2 kB 17.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is inco

In [2]:
import torchvision.transforms.functional as F
import sys
sys.modules["torchvision.transforms.functional_tensor"] = F
print("Patch applied. Safe to import basicsr and realesrgan now.")

Patch applied. Safe to import basicsr and realesrgan now.


# Load All Models

*  RRDBNet = the neural network architecture Real-ESRGAN uses,  RRDB = Residual in Residual Dense Block It's a specific type of network designed for upscaling images, We need to import this to build the model structure before loading weights
*  RealESRGANer = the wrapper class that makes Real-ESRGAN easy to use It handles: loading weights, tiling large images, upscaling, post-processing "tiling" = splitting a large image into small tiles, upscaling each one,then stitching them back together — prevents running out of VRAM
*   BlipProcessor = prepares images before feeding to BLIP model
BlipForConditionalGeneration = the BLIP model that reads images and writes captions


In [23]:
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
token = user_secrets.get_secret("hf_token")
os.environ["hf_token"] = token   
from huggingface_hub import login

# Using the environment variable set above
login(token=os.environ["hf_token"])   


In [3]:
import sys
import torchvision.transforms.functional as F
sys.modules["torchvision.transforms.functional_tensor"] = F
import torch
from basicsr.archs.rrdbnet_arch import RRDBNet
from realesrgan import RealESRGANer   
from diffusers import DiffusionPipeline
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image, ImageFilter
from PIL import ImageDraw
import numpy as np
import os
from huggingface_hub import login, snapshot_download
import urllib


weights_dir = "/kaggle/working/weights"
os.makedirs(weights_dir, exist_ok=True)
weights = f"{weights_dir}/RealESRGAN_x4plus.pth"

if not os.path.exists(weights):
    url = "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth"
    urllib.request.urlretrieve(url, weights)
    
esrga= RRDBNet(
    num_in_ch= 3,
    num_out_ch= 3,
    num_feat= 64,
    num_block= 23,
    num_grow_ch= 32,
    scale= 4
)

enhancer= RealESRGANer(
    scale=4,
    model_path= weights,
    model= esrga,
    tile= 512,# tile=512 means: split the input image into 512×512 tiles
    tile_pad= 10,  # tile_pad=10 = add 10 pixels of overlap between tiles Without overlap, you'd see visible seam lines where tiles meet
    pre_pad=0, # pre_pad = extra padding added before processing
    half= True,  # half=True = use float16 (half precision)
)

print("REAL-ESRGAN Ready")



inpaint = DiffusionPipeline.from_pretrained(
    "sd2-community/stable-diffusion-2-inpainting",
    torch_dtype=torch.bfloat16,
    device_map= "cuda"
)

inpaint.enable_model_cpu_offload()
print("SD2 Inpainting loaded!")


blipprocessor= BlipProcessor.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)

blip_model= BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base",
    torch_dtype= torch.float16
).to("cuda")

print("All Models Ready!")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


REAL-ESRGAN Ready


model_index.json:   0%|          | 0.00/544 [00:00<?, ?B/s]

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

...

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identic

All Models Ready!


# Enhancement Function

In [4]:
import gradio as gr

def enhance_image(image, scale_factor): # scale_factor = string from dropdown: "2x" or "4x"
    if image is None:
        raise gr.Error("Please Upload an Image First..!")

    #convert PIL image to numpy array
    image_array= np.array(image)
    # Real-ESRGAN's enhance() method expects a numpy array, not a PIL image
    # np.array(image) converts PIL → numpy array of shape (height, width, 3)
    # Each pixel is [R, G, B] with values 0-255

    image_array= image_array[:,:,:3]
    # REALESRGAN works only on RGB channels not RGBA(with alpha channel), this slicing helps in keeping first 3 channels only 
    # if a user gives transparent png, this saves from crashing
    outscale= 4 if scale_factor== "4x" else 2

    try:
        output_array, _ = enhancer.enhance(
            image_array,
            outscale= outscale,
        )

    except RuntimeError as e:
        raise gr.Error(f"Enhancement failed: {str(e)}. try smaller image.")
    #convert BGR → RGB before making PIL image
    # output_array[:, :, ::-1] reverses the channel axis
    output_array_rgb = output_array[:, :, ::-1]
    #convert numpy array to PIL image
    output_image= Image.fromarray(output_array)

    # original and after sizes comparison
    original_size= f"{image.width}x{image.height}"
    new_size= f"{output_image.width}x{output_image.height}"
    size_info= f"Original: {original_size} -> enhanced: {new_size}"
    return output_image, size_info

# Outpainting Function

In [5]:
def get_caption(image):
    inputs  = blipprocessor(image.convert("RGB"), return_tensors="pt").to("cuda", torch.float16)
    output  = blip_model.generate(**inputs, max_new_tokens=60)
    caption = blipprocessor.decode(output[0], skip_special_tokens=True)
    return caption


def build_prompt(caption):
    return (
        f"seamless continuation of a scene with {caption}, "
        f"same exact lighting, same exact style, "
        f"same background, extending the existing scene naturally, "
        f"no new objects, no new subjects, only environment continuation"
    )


def extend_one_side(image, direction, pixels, prompt, negative_prompt):
    # ============================================================
    # CORE LOGIC — extends one side by a SMALL number of pixels
    # Called multiple times in small steps instead of one big jump
    # Small steps = SD always has lots of context = coherent output
    # ============================================================
    from PIL import ImageDraw

    orig_w  = image.width
    orig_h  = image.height

    new_w   = orig_w
    new_h   = orig_h
    paste_x = 0
    paste_y = 0

    if direction == "left":
        new_w   = orig_w + pixels
        paste_x = pixels

    elif direction == "right":
        new_w   = orig_w + pixels
        paste_x = 0

    elif direction == "top":
        new_h   = orig_h + pixels
        paste_y = pixels

    elif direction == "bottom":
        new_h   = orig_h + pixels
        paste_y = 0

    # Round to multiple of 8 — SD requirement
    new_w = (new_w // 8) * 8
    new_h = (new_h // 8) * 8

    # Recalculate after rounding
    if direction in ["left", "right"]:
        pixels = new_w - orig_w
    else:
        pixels = new_h - orig_h

    if direction == "left":
        paste_x = pixels
    if direction == "top":
        paste_y = pixels

    # Build black canvas and paste original into it
    canvas = Image.new("RGB", (new_w, new_h), (0, 0, 0))
    canvas.paste(image, (paste_x, paste_y))

    # ── Build mask ────────────────────────────────────────────
    mask = Image.new("L", (new_w, new_h), 255)
    # All white = generate everywhere
    draw = ImageDraw.Draw(mask)

    feather = 30
    # This ensures the black region doesn't touch the very edge of original
    # leaving a thin white strip that GaussianBlur will turn into a gradient


    if direction == "left":
        draw.rectangle([
            paste_x + feather,  feather,
            new_w   - feather,  new_h - feather
        ], fill=0)

    elif direction == "right":
        draw.rectangle([
            feather, feather,
            orig_w  - feather,  new_h - feather
        ], fill=0)

    elif direction == "top":
        draw.rectangle([
            feather,            paste_y + feather,
            new_w   - feather,  new_h   - feather
        ], fill=0)

    elif direction == "bottom":
        draw.rectangle([
            feather,            feather,
            new_w   - feather,  orig_h  - feather
        ], fill=0)

    
    #Apply GaussianBlur to the mask AFTER drawing
    # This is what creates REAL feathering — a soft gradient at the boundary
    mask= mask.filter(ImageFilter.GaussianBlur(radius=30))

    # ── Resize to 512×512 for SD ──────────────────────────────
    sd_size   = 512
    canvas_sd = canvas.resize((sd_size, sd_size), Image.LANCZOS)
    mask_sd   = mask.resize((sd_size, sd_size), Image.LANCZOS)
    # LANCZOS = high quality downsampling, preserves sharp details

    result = inpaint(
        prompt              = prompt,
        image               = canvas_sd,
        mask_image          = mask_sd,
        height              = sd_size,
        width               = sd_size,
        num_inference_steps = 40,
        guidance_scale      = 7.0,
        negative_prompt     = negative_prompt,
    )

    generated_512 = result.images[0]

    generated_full = generated_512.resize((new_w, new_h), Image.LANCZOS)
    return generated_full


def outpaint_image(image, direction, extend_percent, custom_prompt, progress=gr.Progress()):

    if image is None:
        raise gr.Error("Please upload an image first.")

    # Resize input to max 512 on longest side
    max_side = 512
    ratio    = min(max_side / image.width, max_side / image.height)
    
    new_size = (int(image.width * ratio), int(image.height * ratio))
    image    = image.resize(new_size, Image.LANCZOS)

    progress(0.05, desc="Analyzing image with BLIP...")

    if custom_prompt.strip():
        blip_caption = custom_prompt.strip()
    else:
        blip_caption = get_caption(image)

    prompt = build_prompt(blip_caption)

    negative_prompt = (
        "blurry, bad quality, watermark, text, "
        "new person, new face, new subject, extra people, "
        "colorful, vibrant colors, color photography, "
        "duplicate, tiled, repeated pattern, border, frame, "
        "seam, visible edge, abrupt change, inconsistent, "
        "distorted, unnatural, different style, different era"
    )

    # ── KEY CHANGE: small fixed step size, multiple passes ────
    STEP_PX = 64
    # Add only 64 pixels per pass instead of the full extension at once
    # 64px out of a 512px image = 12.5% of canvas = SD has 87.5% context
    # Compare to before: 35% extension = SD only had 65% context
    # More context = SD understands the scene = coherent generation

    # Calculate total pixels needed per direction
    extend   = extend_percent / 100.0
    h_total  = int(image.width  * extend)
    # Total horizontal pixels to add on each side
    v_total  = int(image.height * extend)
    # Total vertical pixels to add on each side

    def make_passes(side, total_px):
        # Break total_px into multiple STEP_PX passes
        # Example: total=180px, STEP_PX=64 → passes of [64, 64, 52]
        passes = []
        remaining = total_px
        while remaining > 0:
            step = min(STEP_PX, remaining)
            # min() = don't overshoot — last step may be smaller
            passes.append((side, step))
            remaining -= step
        return passes
        # Returns list of (direction, pixels) tuples
        # Each tuple = one call to extend_one_side

    # Build full pass list based on chosen direction
    if direction == "Horizontal":
        passes = make_passes("right", h_total) + make_passes("left", h_total)
        # Extend right in small steps, then left in small steps

    elif direction == "Vertical":
        passes = make_passes("bottom", v_total) + make_passes("top", v_total)
        # Bottom first (more grounded content), then top

    else:
        # Both — all four sides in small steps
        passes = (
            make_passes("bottom", v_total) +
            make_passes("top",    v_total) +
            make_passes("right",  h_total) +
            make_passes("left",   h_total)
        )

    total_passes  = len(passes)
    current_image = image

    for i, (side, px) in enumerate(passes):
        progress_val = 0.1 + 0.85 * (i / total_passes)
        progress(progress_val, desc=f"Pass {i+1}/{total_passes} — extending {side} by {px}px")

        current_image = extend_one_side(
            current_image,
            side,
            px,
            prompt,
            negative_prompt
        )
        # Each pass returns a slightly larger image
        # Next pass uses that as input — builds on previous result

    progress(1.0, desc="Done!")

    return (
        current_image,
        f"BLIP caption:\n{blip_caption}\n\nFull prompt:\n{prompt}"
    )

# Gradio Interface

In [6]:
with gr.Blocks(
    title= "Image Enhancer & Outpainter"
) as demo:
    gr.Markdown("# CANVASAI: AI App for Image Enhancement & Outpainting")
    gr.Markdown(
        "**Tab 1** - Enhance: make any image sharper and larger using Real-ESRGAN  \n"
        "**Tab 2** — Outpaint: extend image borders with AI-generated content"
    )
    with gr.Tabs():
        #____________Tab 1- Enhancement____________
        with gr.Tab("Enhance- Super Resolution"):
            gr.Markdown(
                "Upload any image. Real-ESRGAN will sharpen details "
                "and upscale it 2x or 4x."
            )
            with gr.Row():
                with gr.Column(scale=1):
                    enh_input= gr.Image(
                        label= "Upload Image",
                        type= "pil"
                    )
                    enh_scale= gr.Dropdown(
                        choices= ["2x", "4x"],
                        value= "4x",
                        label= "Upscale Factor",
                        info= "4x recommended. Use 2x for very large input images."
                    )
                    enh_btn= gr.Button(
                        "Enhance Image",
                        variant= "primary"
                    )
                with gr.Column(scale=1):
                    enh_output= gr.Image(
                        label= "Enhanced Image",
                        type= "pil",
                        interactive= False
                    )
                    enh_info= gr.Textbox(
                        label= "Size Info",
                        interactive= False
                    )

            enh_btn.click(
                fn= enhance_image,
                inputs= [enh_input, enh_scale],
                outputs= [enh_output, enh_info]
            )
        #__________Outpaint___________________
        with gr.Tab("Outpaint- Extend Image"):
            gr.Markdown(
                "Upload an image. Choose a direction and how much to extend. "
                "BLIP automatically reads your image to guide the generation — "
                "no text prompt needed."
            )
            with gr.Row():
                with gr.Column(scale=1):
                    out_input= gr.Image(
                        label= "Upload Image to Extend",
                        type= "pil",
                        key= "outpaint_image" #key gives this component a unique name
                    )
                    out_direction= gr.Radio(
                        choices= ["Horizontal", "Vertical", "Both"],
                        value= "Horizontal",
                        label= "Extension Direction",
                        info= (
                            "Horizontal = extend left and right  |  "
                            "Vertical = extend top and bottom  |  "
                            "Both = extend all four sides"
                        )
                        
                    )

                    out_percent= gr.Slider(
                        minimum= 10,
                        maximum= 50,
                        value= 25,
                        step=5,
                        label= "Extend by (%)",
                        info= (
                            "How much to add on each side as % of original size. "
                            "25% means each side grows by a quarter of the original."
                        )
                    )
                    out_prompt= gr.Textbox(
                        label= "Custom Prompt (optional)",
                        placeholder= ("Leave empty to auto-detect with BLIP. "
                            "Or type: 'a forest with tall trees and soft sunlight'"),
                        lines=2
                    )
                    out_btn= gr.Button(
                        "Outpaint Image",
                        variant= "primary"
                    )
                with gr.Column(scale=1):
                    out_output= gr.Image(
                        label= "Extended Result",
                        type= "pil",
                        interactive= False,
                    )
                    out_caption= gr.Textbox(
                        label= "prompt used for generation",
                        interactive= False,
                        lines=3,
                    )
            out_btn.click(
                fn= outpaint_image,
                inputs= [out_input, out_direction, out_percent, out_prompt],
                outputs= [out_output, out_caption],
            )

#_______Launch________
demo.launch(
    share= True,
    debug= True,
    
)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://a18a779372c4690149.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


  0%|          | 0/40 [00:00<?, ?it/s]

  0%|          | 0/40 [00:00<?, ?it/s]

  0%|          | 0/40 [00:00<?, ?it/s]

  0%|          | 0/40 [00:00<?, ?it/s]

...

  0%|          | 0/40 [00:00<?, ?it/s]

  0%|          | 0/40 [00:00<?, ?it/s]

  0%|          | 0/40 [00:00<?, ?it/s]

  0%|          | 0/40 [00:00<?, ?it/s]

	Tile 1/4
	Tile 2/4
	Tile 3/4
	Tile 4/4
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://a18a779372c4690149.gradio.live
